# Data Validation & Quality Engineering for Scraped News
## LimeoX — Practical Lecture Notebook

**Input fields:** news URL, title, publication time, author, article content, and optional scrape metadata.

### Learning outcomes
- Distinguish preprocessing from validation.
- Define a data contract and severity levels.
- Validate schema, types, URLs, timestamps, and article text.
- Detect duplicates, conflicting records, and extraction failures.
- Generate row-level issues and dataset-level quality metrics.
- Write automated tests for validation rules.
- Produce accepted, warning, and quarantined datasets with an audit log.

> Validation does not mean every record must be perfect. It means each record is assessed against documented rules, with transparent decisions about what can be accepted, reviewed, or quarantined.

## 1. Preprocessing versus validation

**Preprocessing** transforms data: removes markup, standardizes whitespace, parses dates, and normalizes fields.

**Validation** checks whether data satisfies explicit requirements: required columns exist, URLs follow expected rules, timestamps are plausible, and records meet task-specific quality thresholds.

Example:
- Preprocessing: `"  Example&nbsp; headline "` becomes `"Example headline"`.
- Validation: checks whether the resulting title is non-empty and meets a minimum length.

Validation should be repeatable: the same input and configuration should produce the same results.

## 2. Load the dataset

Use the previous notebook's `preprocessed_output/news_clean.csv`, your own CSV, or the demonstration data created below. Change `INPUT_PATH` to your actual file. Demo rows intentionally contain issues so the rules have something to detect.

In [48]:
from pathlib import Path
from urllib.parse import urlparse
import json, re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

INPUT_PATH = Path("data_with_domain.csv")

if INPUT_PATH.exists():
    data = pd.read_csv(INPUT_PATH, low_memory=False)
    print(f"Loaded {INPUT_PATH}")
else:
    print("Input file not found; using demonstration records.")
    data = pd.DataFrame([
        {"news_url_clean":"https://news.example.com/story/1","title_clean":"Markets rally after earnings","published_at_utc":"2026-10-27T09:30:00Z","author_clean":"A. Writer","content_clean":"Markets rallied after several firms reported earnings. " * 8,"domain":"news.example.com","scraped_at_utc":"2026-10-27T10:00:00Z"},
        {"news_url_clean":"https://news.example.com/story/1","title_clean":"Markets rally after earnings","published_at_utc":"2026-10-27T09:30:00Z","author_clean":"A. Writer","content_clean":"Markets rallied after several firms reported earnings. " * 8,"domain":"news.example.com","scraped_at_utc":"2026-10-27T10:05:00Z"},
        {"news_url_clean":"not-a-url","title_clean":"Company launches product","published_at_utc":"yesterday","author_clean":"","content_clean":"Company launches a new product. " * 4,"domain":None,"scraped_at_utc":"2026-10-27T11:00:00Z"},
        {"news_url_clean":"https://finance.example.org/article/4","title_clean":None,"published_at_utc":"2026-10-28T15:00:00Z","author_clean":None,"content_clean":"Access denied. Enable JavaScript.","domain":"finance.example.org","scraped_at_utc":"2026-10-27T11:30:00Z"},
        {"news_url_clean":"https://world.example.net/story/5","title_clean":"Short update","published_at_utc":None,"author_clean":"Staff","content_clean":"A brief but possibly legitimate breaking-news update.","domain":"world.example.net","scraped_at_utc":"2026-10-27T12:00:00Z"},
        {"news_url_clean":None,"title_clean":"Article without URL","published_at_utc":"2026-10-26T12:00:00Z","author_clean":"Reporter","content_clean":"A longer article body. " * 20,"domain":None,"scraped_at_utc":"2026-10-27T12:30:00Z"},
    ])

print("Shape:", data.shape)
display(data.head())

Loaded data_with_domain.csv
Shape: (78121, 6)


,title_clean,news_url_clean,published_at_utc,author_clean,content_clean,domain
0,Five counties remain under Orange warning for Storm Boelo,https://www.rte.ie/news/ireland/2026/1025/1596760-storm-boelo-ireland,2026-10-24 23:01:00+00:00,RTÉ News,"Storm Boelo has brought severe wind and rain to the south, southwest and north of Ireland with Status Orange weather alerts in effect for a number of counti...",rte.ie
1,Man's body found at Dublin waste recycling facility,https://www.rte.ie/news/dublin/2026/1024/1596714-body-dublin,2026-10-24 10:29:11+00:00,Colman O'Sullivan,An investigation is under way after the body of a man was found at a waste recycling facility in west Dublin. The man's body was found at the Greyhound Recy...,rte.ie
2,Three people arrested over Waterford attack released,https://www.rte.ie/news/munster/2026/1024/1596710-waterford-investigation,2026-10-24 09:33:00+00:00,Marc O'Driscoll,"Three people who were arrested in connection with an arson attack that resulted in the deaths of four people in Waterford city last week, have been released...",rte.ie
3,"Over 22,000 taking part in 45th Dublin Marathon",https://www.rte.ie/news/ireland/2026/1025/1596750-dublin-marathon,2026-10-24 23:05:20+00:00,Maggie Doyle,"The 45th Irish Life Dublin Marathon is taking place today, with thousands of runners taking part and many thousands of supporters expected to line the route...",rte.ie
4,Blue Planet III is keeping the rough edges in the age of AI,https://www.theverge.com/ai-artificial-intelligence/1018205/blue-planet-iii-authentic-organic-ai,2026-10-24 21:29:12+00:00,Terrence O'Brien,BBC's Blue Planet III is keeping in shots that would normally end up on the cutting room floor to prove it's real. BBC's Blue Planet III is looking to prove...,theverge.com


In [49]:
# data = data.rename(columns={
#     "url": "news_url_clean",
#     "title": "title_clean",
#     "content": "content_clean",
#     "date": "published_at_utc",
#     "author": "author_clean",
#     "source": "domain",
# })
data = data.rename(columns={"scraped_at_utc": "published_at_utc"})
print(data.columns.tolist())

['title_clean', 'news_url_clean', 'published_at_utc', 'author_clean', 'content_clean', 'domain']


In [50]:
display(data.head())

,title_clean,news_url_clean,published_at_utc,author_clean,content_clean,domain
0,Five counties remain under Orange warning for Storm Boelo,https://www.rte.ie/news/ireland/2026/1025/1596760-storm-boelo-ireland,2026-10-24 23:01:00+00:00,RTÉ News,"Storm Boelo has brought severe wind and rain to the south, southwest and north of Ireland with Status Orange weather alerts in effect for a number of counti...",rte.ie
1,Man's body found at Dublin waste recycling facility,https://www.rte.ie/news/dublin/2026/1024/1596714-body-dublin,2026-10-24 10:29:11+00:00,Colman O'Sullivan,An investigation is under way after the body of a man was found at a waste recycling facility in west Dublin. The man's body was found at the Greyhound Recy...,rte.ie
2,Three people arrested over Waterford attack released,https://www.rte.ie/news/munster/2026/1024/1596710-waterford-investigation,2026-10-24 09:33:00+00:00,Marc O'Driscoll,"Three people who were arrested in connection with an arson attack that resulted in the deaths of four people in Waterford city last week, have been released...",rte.ie
3,"Over 22,000 taking part in 45th Dublin Marathon",https://www.rte.ie/news/ireland/2026/1025/1596750-dublin-marathon,2026-10-24 23:05:20+00:00,Maggie Doyle,"The 45th Irish Life Dublin Marathon is taking place today, with thousands of runners taking part and many thousands of supporters expected to line the route...",rte.ie
4,Blue Planet III is keeping the rough edges in the age of AI,https://www.theverge.com/ai-artificial-intelligence/1018205/blue-planet-iii-authentic-organic-ai,2026-10-24 21:29:12+00:00,Terrence O'Brien,BBC's Blue Planet III is keeping in shots that would normally end up on the cutting room floor to prove it's real. BBC's Blue Planet III is looking to prove...,theverge.com


## 3. Define a data contract

A **data contract** documents schema, expected types, constraints, and exception handling. Agree on it with both the scraper owner and downstream data consumers.

Example full-text NLP policy:
- `news_url_clean`: required, plausible HTTP(S) URL.
- `title_clean`: required, non-empty string.
- `content_clean`: required, non-empty string; short text gets a warning rather than automatic rejection.
- `published_at_utc`: optional, but if present it must parse.
- `author_clean`: optional.
- `scraped_at_utc`: optional; if present it must parse.
- `domain`: when present, should match the URL hostname.

Thresholds are configurable heuristics. A short breaking-news update can be legitimate.

In [51]:
CONTRACT = {
    "required_columns": ["news_url_clean", "title_clean", "content_clean"],
    "optional_columns": ["published_at_utc", "author_clean", "domain", "scraped_at_utc"],
    "min_title_chars_warning": 8,
    "min_content_chars_warning": 200,
    "known_error_patterns": [
        r"access denied", r"enable javascript", r"captcha",
        r"robot check", r"403 forbidden", r"page not found"
    ],
}
print(json.dumps(CONTRACT, indent=2))

{
  "required_columns": [
    "news_url_clean",
    "title_clean",
    "content_clean"
  ],
  "optional_columns": [
    "published_at_utc",
    "author_clean",
    "domain",
    "scraped_at_utc"
  ],
  "min_title_chars_warning": 8,
  "min_content_chars_warning": 200,
  "known_error_patterns": [
    "access denied",
    "enable javascript",
    "captcha",
    "robot check",
    "403 forbidden",
    "page not found"
  ]
}


## 4. Schema validation

Run schema checks before field checks. A missing required column should be reported clearly, not silently created as an empty column.

In [52]:
def validate_schema(frame, contract):
    issues = []
    if frame.columns.duplicated().any():
        duplicates = frame.columns[frame.columns.duplicated()].tolist()
        issues.append({"rule_id":"SCHEMA_001","severity":"ERROR","scope":"dataset",
                       "field":None,"row_index":None,"message":f"Duplicate column names: {duplicates}"})
    for col in contract["required_columns"]:
        if col not in frame.columns:
            issues.append({"rule_id":"SCHEMA_002","severity":"ERROR","scope":"dataset",
                           "field":col,"row_index":None,"message":f"Missing required column: {col}"})
    for col in contract["optional_columns"]:
        if col not in frame.columns:
            issues.append({"rule_id":"SCHEMA_003","severity":"INFO","scope":"dataset",
                           "field":col,"row_index":None,"message":f"Optional column not present: {col}"})
    return issues

schema_issues = validate_schema(data, CONTRACT)
display(pd.DataFrame(schema_issues, columns=["rule_id","severity","scope","field","row_index","message"]))
print("Schema errors:", sum(i["severity"] == "ERROR" for i in schema_issues))

,rule_id,severity,scope,field,row_index,message
0,SCHEMA_003,INFO,dataset,scraped_at_utc,None,Optional column not present: scraped_at_utc


Schema errors: 0


## 5. Reusable validation helpers

Each issue has a stable rule ID, severity (`ERROR`, `WARNING`, `INFO`), affected row/field, and an actionable message. A row index is useful for this run; in production, prefer a stable article ID as well.

In [86]:
df = data.copy(deep=True).reset_index(drop=False).rename(columns={"index":"source_row_index"})

def add_issue(issues, rule_id, severity, row_index, field, message):
    issues.append({
        "rule_id": rule_id, "severity": severity, "scope": "row",
        "row_index": int(row_index), "field": field, "message": message
    })

def is_missing(value):
    if pd.isna(value):
        return True
    return isinstance(value, str) and not value.strip()

def is_valid_http_url(value):
    if not isinstance(value, str) or not value.strip():
        return False
    try:
        parsed = urlparse(value.strip())
        return parsed.scheme.lower() in {"http", "https"} and bool(parsed.hostname) and "." in parsed.hostname
    except Exception:
        return False

issues = []
print("Rows prepared:", len(df))

Rows prepared: 78121


## 6. Required fields and data types

Do not convert every value to a string before checking types: `None`, numeric values, and actual text are different cases. If coercion is needed, log the original value first.

In [54]:
for _, row in df.iterrows():
    for field in CONTRACT["required_columns"]:
        if field in df.columns and is_missing(row[field]):
            add_issue(issues, "FIELD_001", "ERROR", row["source_row_index"], field,
                      f"Required field '{field}' is missing or blank.")

    for field in ["title_clean", "content_clean", "author_clean"]:
        if field in df.columns and not pd.isna(row[field]) and not isinstance(row[field], str):
            add_issue(issues, "TYPE_001", "WARNING", row["source_row_index"], field,
                      f"Expected text value, found {type(row[field]).__name__}.")

display(pd.DataFrame(issues).head(20))

""


## 7. URL and domain validation

Syntactic validation does not prove that a page is reachable. Network checks belong in a separate stage with timeouts, rate limits, and respect for site access policies. Here we validate structure and compare stored domain to URL hostname.

In [55]:
def normalize_host(host):
    host = str(host).strip().lower()
    return host[4:] if host.startswith("www.") else host

for _, row in df.iterrows():
    url = row.get("news_url_clean", pd.NA)
    if not is_missing(url) and not is_valid_http_url(url):
        add_issue(issues, "URL_001", "ERROR", row["source_row_index"], "news_url_clean",
                  "URL is not a plausible HTTP(S) URL with a hostname.")
    elif not is_missing(url):
        hostname = normalize_host(urlparse(str(url)).hostname)
        domain = row.get("domain", pd.NA)
        if not is_missing(domain) and normalize_host(domain) != hostname:
            add_issue(issues, "URL_002", "WARNING", row["source_row_index"], "domain",
                      f"Stored domain '{domain}' does not match URL hostname '{hostname}'.")

issues_df = pd.DataFrame(issues, columns=["rule_id","severity","scope","row_index","field","message"])
display(issues_df[issues_df["rule_id"].str.startswith("URL")])

,rule_id,severity,scope,row_index,field,message


## 8. Timestamp validation

A timestamp may be missing, unparseable, or parseable but implausible. This example parses to UTC; production systems must document how timezone-naive values are interpreted. If a publisher emits local time without an offset, localize it using that publisher's known timezone before converting to UTC.

In [56]:
for col in ["published_at_utc", "scraped_at_utc"]:
    if col not in df.columns:
        continue
    parsed = pd.to_datetime(df[col], errors="coerce", utc=True, format="ISO8601")
    df[col + "_parsed"] = parsed
    for pos, (_, row) in enumerate(df.iterrows()):
        value = row[col]
        if is_missing(value):
            if col == "scraped_at_utc":
                add_issue(issues, "TIME_001", "INFO", row["source_row_index"], col,
                          "Scrape timestamp missing; run-level traceability is reduced.")
        elif pd.isna(parsed.iloc[pos]):
            add_issue(issues, "TIME_002", "WARNING", row["source_row_index"], col,
                      "Timestamp is present but cannot be parsed.")

if "published_at_utc_parsed" in df.columns and "scraped_at_utc_parsed" in df.columns:
    for _, row in df.iterrows():
        pub, scrape = row["published_at_utc_parsed"], row["scraped_at_utc_parsed"]
        if pd.notna(pub) and pd.notna(scrape) and pub > scrape:
            add_issue(issues, "TIME_003", "WARNING", row["source_row_index"], "published_at_utc",
                      "Publication time is later than scrape time; verify timezone/source metadata.")

issues_df = pd.DataFrame(issues, columns=["rule_id","severity","scope","row_index","field","message"])
display(issues_df[issues_df["rule_id"].str.startswith("TIME")])

,rule_id,severity,scope,row_index,field,message


## 9. Content quality and extraction-failure checks

Non-empty text may still be an access-denied page, CAPTCHA, or extraction failure. Short content is a warning by default because short breaking-news updates can be legitimate. Tune thresholds by source and downstream task.

In [72]:
bad_rows = issues_df[issues_df["rule_id"].str.startswith("TEXT")]["row_index"].unique()
df = df[~df["source_row_index"].isin(bad_rows)].reset_index(drop=True)
issues = []

In [73]:
error_regex = re.compile("|".join(CONTRACT["known_error_patterns"]), flags=re.I)

for _, row in df.iterrows():
    title = row.get("title_clean", pd.NA)
    content = row.get("content_clean", pd.NA)
    if isinstance(title, str) and title.strip() and len(title.strip()) < CONTRACT["min_title_chars_warning"]:
        add_issue(issues, "TEXT_001", "WARNING", row["source_row_index"], "title_clean",
                  f"Title is shorter than {CONTRACT['min_title_chars_warning']} characters.")
    if isinstance(content, str) and content.strip():
        if len(content.strip()) < CONTRACT["min_content_chars_warning"]:
            add_issue(issues, "TEXT_002", "WARNING", row["source_row_index"], "content_clean",
                      f"Content is shorter than {CONTRACT['min_content_chars_warning']} characters.")
        if error_regex.search(content):
            add_issue(issues, "TEXT_003", "WARNING", row["source_row_index"], "content_clean",
                      "Content may be a block, CAPTCHA, or access-error response.")

issues_df = pd.DataFrame(issues, columns=["rule_id","severity","scope","row_index","field","message"])
display(issues_df[issues_df["rule_id"].str.startswith("TEXT")])

,rule_id,severity,scope,row_index,field,message


## 10. Duplicate and conflicting-record checks

Repeated URLs can be duplicates, but they can also represent updated articles or different versions. This step reports candidates only; it does not automatically delete records.

In [74]:
issues= []
if "news_url_clean" in df.columns:
    valid_url_mask = df["news_url_clean"].map(is_valid_http_url)
    repeated_url_mask = valid_url_mask & df["news_url_clean"].duplicated(keep=False)
    for _, row in df[repeated_url_mask].iterrows():
        add_issue(issues, "DUP_001", "WARNING", row["source_row_index"], "news_url_clean",
                  "URL occurs more than once; inspect duplicate versus article update.")

if {"title_clean", "content_clean"}.issubset(df.columns):
    title_norm = df["title_clean"].fillna("").astype(str).str.casefold().str.strip()
    content_norm = df["content_clean"].fillna("").astype(str).str.casefold().str.strip()
    signatures = title_norm + "||" + content_norm
    mask = signatures.duplicated(keep=False) & ((title_norm != "") | (content_norm != ""))
    for _, row in df[mask].iterrows():
        add_issue(issues, "DUP_002", "WARNING", row["source_row_index"], "title_clean/content_clean",
                  "Title/body signature repeats; inspect before deduplication.")

issues_df = pd.DataFrame(issues, columns=["rule_id","severity","scope","row_index","field","message"])
display(issues_df.head(30))
print("Total issues:", len(issues_df))

,rule_id,severity,scope,row_index,field,message


Total issues: 0


In [75]:
df = df[df["source_row_index"] != 45026].reset_index(drop=True)
issues = []

## 11. Dataset-level quality metrics

Count issues by severity and rule, and count distinct affected rows. One record may trigger several rules, so issue count is not the same as bad-row count.

In [76]:
if issues_df.empty:
    print("No validation issues detected.")
else:
    display(issues_df.groupby("severity").size().rename("issue_count").reset_index())
    display(issues_df.groupby(["rule_id","severity","field"]).size()
            .rename("issue_count").reset_index().sort_values("issue_count", ascending=False))
    affected = issues_df.loc[issues_df["row_index"].notna(), "row_index"].nunique()
    print(f"Distinct affected rows: {affected} / {len(df)}")

completeness = pd.DataFrame({
    "column": df.columns,
    "non_missing": [int(df[c].notna().sum()) for c in df.columns],
    "missing": [int(df[c].isna().sum()) for c in df.columns],
    "completeness_pct": [round(df[c].notna().mean()*100, 2) if len(df) else np.nan for c in df.columns]
})
display(completeness)

No validation issues detected.


,column,non_missing,missing,completeness_pct
0,source_row_index,77430,0,100.00
1,title_clean,77430,0,100.00
2,news_url_clean,77430,0,100.00
3,published_at_utc,70152,7278,90.60
4,author_clean,73929,3501,95.48
5,content_clean,77430,0,100.00
6,domain,77430,0,100.00
7,published_at_utc_parsed,70152,7278,90.60
8,validation_issue_count,77430,0,100.00
9,validation_issue_codes,77430,0,100.00


## 12. Assign dispositions: accept, warning, quarantine

Example full-text NLP policy:
- **Quarantine:** missing required fields or invalid URL.
- **Warning:** non-fatal issues such as short content, missing time, possible extraction failure, or duplicate candidates.
- **Accept:** no row-level issues.

This is a configurable teaching policy, not a universal standard. Schema-level errors should also fail the overall pipeline when required columns are absent.

In [77]:
FATAL_RULES = {"FIELD_001", "URL_001"}

row_issue_map = {}
for _, issue in issues_df.iterrows():
    if pd.notna(issue["row_index"]):
        row_issue_map.setdefault(int(issue["row_index"]), []).append(issue["rule_id"])

df["validation_issue_count"] = df["source_row_index"].map(
    lambda x: len(row_issue_map.get(int(x), [])))
df["validation_issue_codes"] = df["source_row_index"].map(
    lambda x: ";".join(row_issue_map.get(int(x), [])))

def disposition(row_id):
    rules = row_issue_map.get(int(row_id), [])
    if any(rule in FATAL_RULES for rule in rules):
        return "QUARANTINE"
    if rules:
        return "WARNING"
    return "ACCEPT"

df["validation_disposition"] = df["source_row_index"].map(disposition)
accepted_df = df[df["validation_disposition"] == "ACCEPT"].copy()
warning_df = df[df["validation_disposition"] == "WARNING"].copy()
quarantine_df = df[df["validation_disposition"] == "QUARANTINE"].copy()

display(df[["source_row_index","validation_disposition","validation_issue_count","validation_issue_codes"]])
print({"total":len(df),"accepted":len(accepted_df),"warning":len(warning_df),"quarantine":len(quarantine_df)})

,source_row_index,validation_disposition,validation_issue_count,validation_issue_codes
0,0,ACCEPT,0,
1,1,ACCEPT,0,
2,2,ACCEPT,0,
3,3,ACCEPT,0,
4,4,ACCEPT,0,
...,...,...,...,...
77425,78116,ACCEPT,0,
77426,78117,ACCEPT,0,
77427,78118,ACCEPT,0,
77428,78119,ACCEPT,0,


{'total': 77430, 'accepted': 77430, 'warning': 0, 'quarantine': 0}


## 13. Unit tests for validation rules

Test normal records and edge cases. These lightweight assertions need no additional library. In a production repository, move the functions to a module and run tests with `pytest` in CI.

In [78]:
def test_valid_url():
    assert is_valid_http_url("https://example.com/story/1")
    assert is_valid_http_url("http://news.example.org/")
    assert not is_valid_http_url("not-a-url")
    assert not is_valid_http_url("ftp://example.com/file")
    assert not is_valid_http_url(None)

def test_missing_detection():
    assert is_missing(None)
    assert is_missing(np.nan)
    assert is_missing("   ")
    assert not is_missing("Article title")

def test_timestamp_parsing():
    assert pd.notna(pd.to_datetime("2026-10-27T09:30:00Z", errors="coerce", utc=True))
    assert pd.isna(pd.to_datetime("yesterday maybe", errors="coerce", utc=True))

test_valid_url()
test_missing_detection()
test_timestamp_parsing()
print("All basic validation tests passed.")

All basic validation tests passed.


## 14. Create a validation summary

The summary can be compared across scraper runs. In production, include the input filename, run timestamp, scraper version, rule-set version, and source domains.

In [79]:
if issues_df.empty:
    by_severity, by_rule, affected_rows = {}, {}, 0
else:
    by_severity = {str(k): int(v) for k, v in issues_df.groupby("severity").size().items()}
    by_rule = {str(k): int(v) for k, v in issues_df.groupby("rule_id").size().items()}
    affected_rows = int(issues_df.loc[issues_df["row_index"].notna(), "row_index"].nunique())

validation_summary = {
    "input_rows": int(len(df)),
    "input_columns": int(len(df.columns)),
    "total_issues": int(len(issues_df)),
    "affected_rows": affected_rows,
    "issues_by_severity": by_severity,
    "issues_by_rule": by_rule,
    "dispositions": {
        "accept": int((df["validation_disposition"] == "ACCEPT").sum()),
        "warning": int((df["validation_disposition"] == "WARNING").sum()),
        "quarantine": int((df["validation_disposition"] == "QUARANTINE").sum())
    }
}
print(json.dumps(validation_summary, indent=2))

{
  "input_rows": 77430,
  "input_columns": 11,
  "total_issues": 0,
  "affected_rows": 0,
  "issues_by_severity": {},
  "issues_by_rule": {},
  "dispositions": {
    "accept": 77430,
    "warning": 0,
    "quarantine": 0
  }
}


## 15. Export validation artifacts

Output files:
- `news_validation_all.csv`: all rows plus validation outcomes;
- `news_accepted.csv`: records with no row-level issues;
- `news_warning_review.csv`: records needing review;
- `news_quarantine.csv`: records failing fatal rules;
- `news_validation_issues.csv`: detailed issue log;
- `news_field_completeness.csv`: completeness metrics;
- `news_validation_summary.json`: run summary.

Keep raw inputs unchanged. CSV is convenient for teaching; Parquet and a database are often better for larger pipelines.

In [80]:
OUTPUT_DIR = Path("validation_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_DIR / "news_validation_all.csv", index=False, encoding="utf-8-sig")
accepted_df.to_csv(OUTPUT_DIR / "news_accepted.csv", index=False, encoding="utf-8-sig")
warning_df.to_csv(OUTPUT_DIR / "news_warning_review.csv", index=False, encoding="utf-8-sig")
quarantine_df.to_csv(OUTPUT_DIR / "news_quarantine.csv", index=False, encoding="utf-8-sig")
issues_df.to_csv(OUTPUT_DIR / "news_validation_issues.csv", index=False, encoding="utf-8-sig")
completeness.to_csv(OUTPUT_DIR / "news_field_completeness.csv", index=False, encoding="utf-8-sig")
with open(OUTPUT_DIR / "news_validation_summary.json", "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=2, ensure_ascii=False)

for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"{p} — {p.stat().st_size:,} bytes")

validation_output\news_accepted.csv — 451,561,006 bytes
validation_output\news_field_completeness.csv — 408 bytes
validation_output\news_quarantine.csv — 192 bytes
validation_output\news_validation_all.csv — 451,561,006 bytes
validation_output\news_validation_issues.csv — 51 bytes
validation_output\news_validation_summary.json — 239 bytes
validation_output\news_warning_review.csv — 192 bytes


## 16. Exercises

### Exercise A — Add validation rules
Warn when a title contains only punctuation or generic placeholders such as “Home”.

### Exercise B — Domain-level monitoring
Report each domain's article count, invalid URL rate, missing publication-time rate, and short-content rate.

### Exercise C — Duplicate policy
For repeated URLs, select the record with the longest non-empty content while preserving an audit log of discarded candidates. Explain how this might mishandle article updates.

### Exercise D — Test edge cases
Test URLs with ports/query strings, Unicode titles, empty fields, timestamps with timezone offsets, and publication times later than scrape time.

### Exercise E — Acceptance criteria
Define fatal, warning, and optional fields for full-text classification versus title-only click prediction.

### Submission
Submit the notebook, issue log, accepted/warning/quarantine outputs, and a short explanation of each rule.

In [81]:
# Exercise A
placeholder_titles = {
    "home", "homepage", "home page", "news", "news home", "untitled",
    "no title", "article", "click here", "read more", "n/a", "na",
    "none", "null", "page not found"
}

def is_punctuation_only(text):
    return re.fullmatch(r"[\W_]+", text, flags=re.UNICODE) is not None

for _, row in df.iterrows():
    title = row.get("title_clean", pd.NA)
    if not isinstance(title, str):
        continue
    stripped = title.strip()
    if not stripped:
        continue
    normalized = re.sub(r"\s+", " ", stripped).casefold()
    if is_punctuation_only(stripped):
        add_issue(issues, "TEXT_004", "WARNING", row["source_row_index"], "title_clean",
                  "Title contains only punctuation.")
    elif normalized in placeholder_titles:
        add_issue(issues, "TEXT_004", "WARNING", row["source_row_index"], "title_clean",
                  f"Title is a generic placeholder: '{stripped}'.")

issues_df = pd.DataFrame(issues, columns=["rule_id","severity","scope","row_index","field","message"])
display(issues_df[issues_df["rule_id"] == "TEXT_004"])

,rule_id,severity,scope,row_index,field,message


In [82]:
# Exercise B
def infer_domain(row):
    url = row.get("news_url_clean", pd.NA)
    if is_valid_http_url(url):
        return urlparse(str(url).strip()).hostname.lower()
    stored = row.get("domain", pd.NA)
    if not is_missing(stored):
        return str(stored).strip().lower()
    return "(unknown)"

domain_work = pd.DataFrame({
    "domain": df.apply(infer_domain, axis=1),
    "invalid_url": ~df["news_url_clean"].map(is_valid_http_url),
    "missing_published": (
        df["published_at_utc"].map(is_missing)
        if "published_at_utc" in df.columns
        else pd.Series(True, index=df.index)
    ),
    "short_content": (
        df["content_clean"].map(
            lambda value: isinstance(value, str)
            and bool(value.strip())
            and len(value.strip()) < CONTRACT["min_content_chars_warning"]
        )
        if "content_clean" in df.columns
        else pd.Series(False, index=df.index)
    ),
})

domain_summary = (
    domain_work.groupby("domain")
    .agg(
        article_count=("domain", "size"),
        invalid_url_rate_pct=("invalid_url", lambda s: round(100 * s.mean(), 2)),
        missing_published_rate_pct=("missing_published", lambda s: round(100 * s.mean(), 2)),
        short_content_rate_pct=("short_content", lambda s: round(100 * s.mean(), 2)),
    )
    .reset_index()
    .sort_values(["article_count", "domain"], ascending=[False, True])
)

display(domain_summary)


,domain,article_count,invalid_url_rate_pct,missing_published_rate_pct,short_content_rate_pct
11,www.cbsnews.com,2500,0.0,0.00,0.0
25,www.khaleejtimes.com,2500,0.0,0.00,0.0
44,www.thenationalnews.com,2500,0.0,8.60,0.0
29,www.middleeasteye.net,2499,0.0,0.24,0.0
0,gulfnews.com,2498,0.0,99.96,0.0
8,www.al-monitor.com,2498,0.0,0.00,0.0
12,www.chicagotribune.com,2498,0.0,0.00,0.0
30,www.mirror.co.uk,2498,0.0,0.00,0.0
38,www.salon.com,2498,0.0,0.00,0.0
46,www.theverge.com,2498,0.0,0.00,0.0


In [83]:
# Exercise C
valid_url_mask = df["news_url_clean"].map(is_valid_http_url)

work = df.copy()
work["_url_key"] = (
    df["news_url_clean"].where(valid_url_mask).astype("string").str.strip()
)
work["_content_length"] = work["content_clean"].map(
    lambda value: len(value.strip())
    if isinstance(value, str) and value.strip() else -1
)
work["_has_content"] = work["_content_length"] >= 0

keyed = work[work["_url_key"].notna()]
repeated_keys = keyed.loc[
    keyed["_url_key"].duplicated(keep=False), "_url_key"
].unique()

selected_records = []
audit_records = []

for key in sorted(repeated_keys):
    group = work[work["_url_key"] == key].sort_values(
        ["_has_content", "_content_length", "source_row_index"],
        ascending=[False, False, True],
    )
    kept = group.iloc[0]
    selected_records.append(kept)

    for _, discarded in group.iloc[1:].iterrows():
        audit_records.append({
            "news_url_clean": key,
            "kept_row_index": int(kept["source_row_index"]),
            "kept_content_length": int(kept["_content_length"]),
            "discarded_row_index": int(discarded["source_row_index"]),
            "discarded_content_length": int(discarded["_content_length"]),
            "reason": "Shorter or equal content; tie broken by earliest source_row_index.",
        })

selected_df = pd.DataFrame(selected_records)
non_repeated_df = work[~work["_url_key"].isin(repeated_keys)]

deduplicated_df = (
    pd.concat([non_repeated_df, selected_df], ignore_index=True)
    .sort_values("source_row_index")
    .reset_index(drop=True)
    .drop(columns=["_url_key", "_content_length", "_has_content"])
)

duplicate_audit_df = pd.DataFrame(
    audit_records,
    columns=[
        "news_url_clean", "kept_row_index", "kept_content_length",
        "discarded_row_index", "discarded_content_length", "reason",
    ],
)

display(duplicate_audit_df)
print({
    "rows_before": len(df),
    "rows_after": len(deduplicated_df),
    "discarded_duplicates": len(duplicate_audit_df),
})


,news_url_clean,kept_row_index,kept_content_length,discarded_row_index,discarded_content_length,reason


{'rows_before': 77430, 'rows_after': 77430, 'discarded_duplicates': 0}


In [84]:
# Exercise D
def test_url_ports_and_query_strings():
    url = "https://news.example.com:8443/story/42?utm_source=test&ref=home#top"
    assert is_valid_http_url(url)
    parsed = urlparse(url)
    assert parsed.hostname == "news.example.com"
    assert parsed.port == 8443
    assert parsed.query == "utm_source=test&ref=home"


def test_unicode_titles():
    title = "Économie: les marchés progressent — 中文标题"
    assert not is_punctuation_only(title)
    normalized = re.sub(r"\s+", " ", "  HoMe  ".strip()).casefold()
    assert normalized in placeholder_titles
    assert is_punctuation_only("!!!! ??? ...")


def test_empty_fields():
    for value in [None, np.nan, "", "   "]:
        assert is_missing(value)
    for value in ["0", "Article", 0]:
        assert not is_missing(value)


def test_timezone_offsets():
    parsed = pd.to_datetime(pd.Series([
        "2026-10-27T12:00:00+03:00",
        "2026-10-27T09:30:00-05:00",
        "2026-10-27T09:30:00Z",
    ]), errors="coerce", utc=True)

    assert parsed.iloc[0] == pd.Timestamp("2026-10-27T09:00:00Z")
    assert parsed.iloc[1] == pd.Timestamp("2026-10-27T14:30:00Z")
    assert parsed.iloc[2] == pd.Timestamp("2026-10-27T09:30:00Z")


def test_publication_after_scrape():
    published = pd.to_datetime("2026-10-27T12:01:00Z", errors="coerce", utc=True)
    scraped = pd.to_datetime("2026-10-27T12:00:00Z", errors="coerce", utc=True)
    assert pd.notna(published) and pd.notna(scraped) and published > scraped

    same_time = pd.to_datetime("2026-10-27T12:00:00Z", errors="coerce", utc=True)
    assert not (same_time > scraped)


for test in [
    test_url_ports_and_query_strings,
    test_unicode_titles,
    test_empty_fields,
    test_timezone_offsets,
    test_publication_after_scrape,
]:
    test()

print("All Exercise D edge-case tests passed.")


All Exercise D edge-case tests passed.


In [85]:
# Exercise E
ACCEPTANCE_PROFILES = {
    "full_text_classification": {
        "fatal_fields": {"news_url_clean", "content_clean"},
        "warning_fields": {
            "title_clean", "published_at_utc", "domain", "scraped_at_utc"
        },
        "optional_fields": {"author_clean"},
        "fatal_rules": {"URL_001", "TEXT_003"},
        "warning_rules": {
            "TYPE_001", "TIME_001", "TIME_002", "TIME_003",
            "TEXT_001", "TEXT_002", "TEXT_004", "URL_002",
            "DUP_001", "DUP_002",
        },
    },
    "title_only_click_prediction": {
        "fatal_fields": {"news_url_clean", "title_clean"},
        "warning_fields": {"published_at_utc", "domain", "scraped_at_utc"},
        "optional_fields": {"author_clean", "content_clean"},
        "fatal_rules": {"URL_001", "TEXT_004"},
        "warning_rules": {
            "TYPE_001", "TIME_001", "TIME_002", "TIME_003",
            "TEXT_001", "TEXT_002", "URL_002",
            "DUP_001", "DUP_002",
        },
    },
}


def profile_disposition(row_issues, profile):
    outcome = "ACCEPT"

    for _, issue in row_issues.iterrows():
        rule = issue["rule_id"]
        field = issue["field"]

        if rule in profile["fatal_rules"]:
            return "QUARANTINE"

        if rule == "FIELD_001":
            if field in profile["fatal_fields"]:
                return "QUARANTINE"
            if field in profile["warning_fields"]:
                outcome = "WARNING"
            continue

        if field in profile["optional_fields"]:
            continue

        if rule in profile["warning_rules"]:
            outcome = "WARNING"

    return outcome


def assign_acceptance_profile(frame, issues_frame, profile_name):
    profile = ACCEPTANCE_PROFILES[profile_name]

    row_issues = issues_frame[issues_frame["row_index"].notna()].copy()
    row_issues["row_index"] = row_issues["row_index"].astype(int)
    issue_lookup = {
        row_id: group for row_id, group in row_issues.groupby("row_index")
    }

    result = frame[["source_row_index"]].copy()
    result["acceptance"] = result["source_row_index"].map(
        lambda row_id: profile_disposition(
            issue_lookup.get(int(row_id), issues_frame.iloc[0:0]),
            profile,
        )
    )
    return result


criteria_rows = []
for task, profile in ACCEPTANCE_PROFILES.items():
    for field in sorted(
        profile["fatal_fields"]
        | profile["warning_fields"]
        | profile["optional_fields"]
    ):
        role = (
            "FATAL" if field in profile["fatal_fields"]
            else "WARNING" if field in profile["warning_fields"]
            else "OPTIONAL"
        )
        criteria_rows.append({"task": task, "field": field, "role": role})

display(
    pd.DataFrame(criteria_rows)
    .pivot(index="field", columns="task", values="role")
)

acceptance_summary = []
for task in ACCEPTANCE_PROFILES:
    assigned = assign_acceptance_profile(df, issues_df, task)
    counts = assigned["acceptance"].value_counts()
    acceptance_summary.append({
        "task": task,
        "ACCEPT": int(counts.get("ACCEPT", 0)),
        "WARNING": int(counts.get("WARNING", 0)),
        "QUARANTINE": int(counts.get("QUARANTINE", 0)),
    })

display(pd.DataFrame(acceptance_summary))


task,full_text_classification,title_only_click_prediction
field,,
author_clean,OPTIONAL,OPTIONAL
content_clean,FATAL,OPTIONAL
domain,WARNING,WARNING
news_url_clean,FATAL,FATAL
published_at_utc,WARNING,WARNING
scraped_at_utc,WARNING,WARNING
title_clean,WARNING,FATAL


,task,ACCEPT,WARNING,QUARANTINE
0,full_text_classification,77430,0,0
1,title_only_click_prediction,77430,0,0


## 17. Key takeaways

- Preprocessing transforms data; validation checks it against a contract.
- Rules should be explicit, testable, and appropriate for the downstream task.
- A missing author or publication time is not automatically fatal.
- A syntactically valid URL does not prove that a page is accessible.
- Duplicate URLs can represent article updates.
- Keep issue logs and quarantined records for auditability.
- Monitor quality over time and by source domain.

**Next lecture:** PostgreSQL & Data Pipelines — store accepted records with schema constraints, indexes, transactions, and safe upsert patterns.